# MC MOD Translator

Minecraft の Mod を日本語化するノートブックです。ランタイムのタイプを T4 GPU にしてから、セル 1 から 4 を順に実行してください。

- Mod、翻訳結果、翻訳の途中経過は Google ドライブの `DRIVE_DIR` に保存します。
- ngrok を使う場合は、シークレットに `NGROK_AUTHTOKEN` を登録してください。

## 中断した翻訳を再開する

1. ランタイムに再接続し、セル 1 から 4 をもう一度実行します。
2. `AUTO_RESUME` がオンの場合は自動で再開します。オフの場合は、画面上部の「再開」を押してください。


In [ ]:
#@title 1. 設定
REPO_URL = "https://github.com/YOUR_NAME/mc-mod-translator"  #@param {type:"string"}
TUNNEL = "ngrok"  #@param ["ngrok", "cloudflared", "colab"]
#@markdown - ngrok: シークレット `NGROK_AUTHTOKEN` が必要
#@markdown - cloudflared: アカウント登録が不要
#@markdown - colab: ノートブックを開いている本人だけが接続できる
DRIVE_DIR = "MyDrive/mc-mod-translator"  #@param {type:"string"}
#@markdown Google ドライブ上の保存先
AUTO_RESUME = True  #@param {type:"boolean"}
#@markdown 中断した翻訳を起動時に再開する
CACHE_LLAMA_TO_DRIVE = True  #@param {type:"boolean"}
#@markdown ビルドした llama.cpp をドライブに保存し、2 回目以降のセットアップを短くする


In [ ]:
#@title 2. セットアップ (初回は 10〜15 分)
import os, subprocess, shutil
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU が見つかりません。ランタイムのタイプを T4 GPU に変更してください。"
%cd /content
if not os.path.exists("/content/app/run.py"):
    if os.path.exists("/content/mc-mod-translator.zip"):           # ZIP を使う場合は /content/ にアップロードしておく
        !unzip -q -o /content/mc-mod-translator.zip -d /content/app
    else:
        !git clone --depth 1 {REPO_URL} /content/app
%cd /content/app
!pip -q install -r requirements.txt

# 保存先を Google ドライブに置く
from google.colab import drive
drive.mount("/content/drive")
WORKDIR = "/content/drive/" + DRIVE_DIR.strip("/")
os.makedirs(WORKDIR, exist_ok=True)
os.environ["MCT_WORKDIR"] = WORKDIR
os.environ["MCT_CACHEDIR"] = "/content/hf_cache"   # モデルはドライブではなく Colab のディスクに置く
print("保存先:", WORKDIR)

LLAMA_DIR = "/content/llama.cpp"
if CACHE_LLAMA_TO_DRIVE:
    cached = WORKDIR + "/cache/llama-server"
    if os.path.exists(cached):
        os.makedirs(f"{LLAMA_DIR}/build/bin", exist_ok=True)
        shutil.copy(cached, f"{LLAMA_DIR}/build/bin/llama-server"); os.chmod(f"{LLAMA_DIR}/build/bin/llama-server", 0o755)
!bash setup_llama.sh {LLAMA_DIR}
if CACHE_LLAMA_TO_DRIVE:
    os.makedirs(WORKDIR + "/cache", exist_ok=True)
    shutil.copy(f"{LLAMA_DIR}/build/bin/llama-server", WORKDIR + "/cache/llama-server")
os.environ["LLAMA_SERVER"] = f"{LLAMA_DIR}/build/bin/llama-server"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
try:
    from google.colab import userdata
    for k in ("NGROK_AUTHTOKEN", "HF_TOKEN"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
except ImportError: pass
print("セットアップが完了しました")

import json, glob
pending = [j for j in (json.load(open(p)) for p in glob.glob(WORKDIR + "/jobs/*/job.json"))
           if j["status"] in ("running", "interrupted", "failed", "cancelled")]
for j in pending:
    u = j["units"].values()
    print(f"中断した翻訳 {j['id']} ({j['status']}): {sum(x['done'] for x in u)}/{sum(x['total'] for x in u)} 件")


In [ ]:
#@title 3. 接続の維持
#@markdown 操作がない状態が続いたときの切断を防ぎます。ブラウザのタブは開いたままにしてください。
#@markdown 無料版の利用時間 (最大約 12 時間) と GPU の利用枠の上限は変わりません。
from IPython.display import display, Javascript
display(Javascript('''
if (!window.__mctKeepAlive) {
  window.__mctKeepAlive = setInterval(() => {
    const b = document.querySelector("colab-connect-button");
    const inner = b && b.shadowRoot && b.shadowRoot.querySelector("#connect");
    if (inner) inner.click(); else if (b) b.click();
    document.dispatchEvent(new MouseEvent("mousemove", {bubbles: true}));
    console.log("[mct] keep-alive", new Date().toLocaleTimeString());
  }, 60 * 1000);
}
'''))
print("接続の維持を開始しました")


In [ ]:
#@title 4. 起動 (実行したままにしてください)
%cd /content/app
RESUME_ARG = "--resume latest" if AUTO_RESUME else ""
%run run.py --tunnel {TUNNEL} --workdir "{WORKDIR}" {RESUME_ARG}
